[Reference](https://medium.com/@GaoDalie_AI/deepseek-v4-turbovec-rag-better-ocr-self-hosted-80d5c615aaab$0)

In [1]:
# query.py
from turbovec import TurboQuantIndex
from langchain_ollama import OllamaEmbeddings
from ollama import chat

embedder = OllamaEmbeddings(model="bge-m3:latest")

index = TurboQuantIndex.load("index.tq")

query = "What are the company’s total assets as of December 31, 2018?"
q_vec = [embedder.embed_query(query)]

scores, indices = index.search(q_vec, k=3)

with open("data.txt") as f:
    texts = f.readlines()

retrieved = [texts[i].strip() for i in indices[0]]
retrieved = list(dict.fromkeys(retrieved))

context = "\n".join(retrieved)

print("\n🔍 Retrieved Context:\n")
for i, chunk in enumerate(retrieved, 1):
    print(f"{i}. {chunk}")

response = chat(
    model='deepseek-v4-flash:cloud',
    messages=[
        {
            "role": "system",
            "content": """You are a strict assistant.

Rules:
1. ONLY use the provided context.
2. DO NOT use prior knowledge.
3. If the answer is not fully in the context, say exactly: "Not found in context".
"""
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
        }
    ]
)

print("\n💡 Answer:\n")
print(response['message']['content'])

In [2]:
# indexer.py
from turbovec import TurboQuantIndex
from langchain_ollama import OllamaEmbeddings

embedder = OllamaEmbeddings(model="bge-m3:latest")

with open("data/docs.txt") as f:
    texts = [line.strip() for line in f.readlines() if line.strip()]

vectors = embedder.embed_documents(texts)

index = TurboQuantIndex(dim=len(vectors[0]), bit_width=4)
index.add(vectors)

index.write("index/index.tq")

print("✅ Index built successfully!")

In [3]:
# rag.py
from turbovec import TurboQuantIndex
from langchain_ollama import OllamaEmbeddings
from ollama import chat
import numpy as np
embedder = OllamaEmbeddings(model="bge-m3:latest")

def build_index(texts):
    vectors = embedder.embed_documents(texts)
    vectors = np.array(vectors, dtype=np.float32)  # add this
    index = TurboQuantIndex(dim=vectors.shape[1], bit_width=4)
    index.add(vectors)
    return index

def ask(query, index, texts, k=3, model="deepseek-v4-flash"):
    q_vec = np.array([embedder.embed_query(query)], dtype=np.float32)
    scores, indices = index.search(q_vec, k=k)
    retrieved = [texts[i] for i in indices[0]]
    context = "\n".join(retrieved)

    response = chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": """STRICT MODE:
- Answer ONLY from context
- No external knowledge
- If missing: say 'Not found in context'
"""
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ]
    )

    return response['message']['content'], retrieved